> **対応するブログ記事**: [#8 Welch's t-testとVolcanoプロットで差分発現タンパク質を見つける](../blog/article-08-differential.md)
>
> このNotebookはブログ記事 #8 のコードをセルごとに実行できるインタラクティブ版です。Welch t検定やVolcanoプロットの詳しい解説はブログ記事を参照してください。

# Step 8: 差分発現解析（Figure 2）

In [ ]:
import numpy as np              # 数値計算ライブラリ（配列操作・数学関数）
import pandas as pd              # データフレーム操作ライブラリ（表形式データの読み書き・加工）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（図の作成・保存）
from matplotlib.patches import Ellipse        # 楕円パッチ（信頼楕円の描画に使用）
import matplotlib.transforms as transforms    # 座標変換（楕円のスケーリング・回転に使用）
import seaborn as sns            # 統計可視化ライブラリ（ヒートマップ・クラスタリング描画）
from scipy import stats          # 統計検定ライブラリ（Welch t検定に使用）
from sklearn.decomposition import PCA  # 主成分分析（次元削減・サンプル分布の可視化）

# Jupyter上でグラフをインライン表示するマジックコマンド
%matplotlib inline

In [ ]:
# --- パス設定 ---
RESULTS  = "../results"            # 解析結果の出力先ディレクトリ
FIG_DIR  = f"{RESULTS}/figures"    # 図の保存先ディレクトリ
TABLE_DIR = f"{RESULTS}/tables"    # 表（CSV）の保存先ディレクトリ

# --- 統計検定の閾値 ---
P_THRESHOLD    = 0.05    # p値の有意水準（5%）：これ未満を統計的に有意とする
LOG2FC_THRESHOLD = 1.0   # log2 fold changeの閾値（2倍変化に相当）：発現量変化の大きさの基準

# --- 配色設定 ---
NORMAL, TUMOR = "#3498DB", "#E74C3C"      # Normal（青）とTumor（赤）のサンプル色
UP, DOWN, NS  = "#E74C3C", "#3498DB", "#CCCCCC"  # 発現上昇（赤）、低下（青）、非有意（灰）の色
COND_MAP = {"Normal": NORMAL, "Tumor": TUMOR}    # 条件名から色への変換辞書

In [ ]:
# データ読み込み & Normal/Tumor サンプル分離

# 前処理済みのタンパク質発現量マトリクスを読み込む（行:タンパク質、列:サンプル）
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)
# サンプル情報（サンプル名・条件など）を読み込む
sample_info = pd.read_csv(f"{RESULTS}/sample_info.csv")
# サンプル名をインデックスにして条件（Normal/Tumor）を取得するSeriesを作成
conditions = sample_info.set_index("Sample")["Condition"]

# Normal条件のサンプル名をリストとして取得
normal_samples = sample_info.query("Condition == 'Normal'")["Sample"].tolist()
# Tumor条件のサンプル名をリストとして取得
tumor_samples  = sample_info.query("Condition == 'Tumor'")["Sample"].tolist()
# データセットの概要を表示：タンパク質数、Normal・Tumorサンプル数
print(f"Proteins: {len(df)}, Normal: {len(normal_samples)}, Tumor: {len(tumor_samples)}")

## Welch's t-test

In [ ]:
def welch_ttest(df, normal_samples, tumor_samples):
    """全タンパク質に Welch t-test を実行し DataFrame を返す。"""
    rows = []  # 各タンパク質の検定結果を格納するリスト
    for protein in df.index:
        # 各タンパク質について、Normal群の発現値を取得（欠損値は除外）
        nv = df.loc[protein, normal_samples].dropna()
        # Tumor群の発現値を取得（欠損値は除外）
        tv = df.loc[protein, tumor_samples].dropna()
        # サンプル数が2未満の群がある場合はt検定できないのでスキップ
        if len(nv) < 2 or len(tv) < 2:
            continue
        # Welch t検定を実行（等分散を仮定しない）：t統計量とp値を取得
        t_stat, p_val = stats.ttest_ind(tv, nv, equal_var=False)
        # log2 fold change = Tumor群の平均 - Normal群の平均（log2スケールでの差分）
        log2fc = tv.mean() - nv.mean()
        # 検定結果を辞書として追加
        rows.append({
            "Protein": protein,            # タンパク質名
            "Mean_Normal": nv.mean(),      # Normal群の平均発現量
            "Mean_Tumor": tv.mean(),       # Tumor群の平均発現量
            "Log2FC": log2fc,              # log2 fold change（正:Tumorで上昇、負:低下）
            "T_statistic": t_stat,         # t統計量（群間差の大きさを表す）
            "P_value": p_val,              # p値（帰無仮説が正しい確率、小さいほど有意）
            # -log10(p値)：Volcanoプロットのy軸用（大きいほど有意）
            # 1e-300でクリップしてlog(0)のエラーを防ぐ
            "Neg_log10_P": -np.log10(max(p_val, 1e-300)),
        })
    # リストからDataFrameを作成
    result = pd.DataFrame(rows)
    # 有意性の分類：デフォルトは"NS"（Not Significant = 非有意）
    result["Significant"] = "NS"
    # p値が閾値未満 かつ log2FCが正の閾値超 → "Up"（Tumorで有意に上昇）
    result.loc[(result["P_value"] < P_THRESHOLD) & (result["Log2FC"] >  LOG2FC_THRESHOLD), "Significant"] = "Up"
    # p値が閾値未満 かつ log2FCが負の閾値未満 → "Down"（Tumorで有意に低下）
    result.loc[(result["P_value"] < P_THRESHOLD) & (result["Log2FC"] < -LOG2FC_THRESHOLD), "Significant"] = "Down"
    return result  # 全タンパク質の検定結果DataFrameを返す

In [ ]:
# 全タンパク質に対してWelch t検定を実行し、結果をDataFrameに格納
result_df = welch_ttest(df, normal_samples, tumor_samples)

# 有意に上昇（Up）したタンパク質の数をカウント
n_up   = (result_df["Significant"] == "Up").sum()
# 有意に低下（Down）したタンパク質の数をカウント
n_down = (result_df["Significant"] == "Down").sum()
# 検定したタンパク質の総数を表示
print(f"検定タンパク質数: {len(result_df)}")
# 有意差ありのタンパク質数（上昇+低下）とその内訳を表示
print(f"有意差あり: {n_up + n_down} (Up {n_up}, Down {n_down})")

# 検定結果をCSVファイルとして保存（後続の解析で使用）
result_df.to_csv(f"{TABLE_DIR}/differential_proteins.csv", index=False)

## Volcano プロット

In [ ]:
# Volcanoプロットの作成（横軸:log2FC、縦軸:-log10(p値)）
fig, ax = plt.subplots(figsize=(8, 6))  # 8x6インチのFigureとAxesを作成

# 各有意性カテゴリごとに散布図をプロット（NS→Up→Downの順で描画）
for sig, color, alpha in [("NS", NS, 0.3), ("Up", UP, 0.6), ("Down", DOWN, 0.6)]:
    m = result_df["Significant"] == sig  # 該当カテゴリのブールマスクを作成
    # 散布図を描画：x=log2FC, y=-log10(p値), s=点のサイズ, alpha=透明度
    ax.scatter(result_df.loc[m, "Log2FC"], result_df.loc[m, "Neg_log10_P"],
              c=color, s=10, alpha=alpha, label=f"{sig} ({m.sum()})")

# p値の閾値を水平破線で描画（これより上が統計的に有意）
ax.axhline(-np.log10(P_THRESHOLD), color="gray", ls="--", lw=0.5)
# log2FCの正の閾値を垂直破線で描画（これより右がUp）
ax.axvline( LOG2FC_THRESHOLD, color="gray", ls="--", lw=0.5)
# log2FCの負の閾値を垂直破線で描画（これより左がDown）
ax.axvline(-LOG2FC_THRESHOLD, color="gray", ls="--", lw=0.5)

# x軸ラベル：Tumor対Normalのlog2 fold change
ax.set_xlabel("Log2 Fold Change (Tumor / Normal)")
# y軸ラベル：p値の負のlog10変換（大きいほど有意）
ax.set_ylabel("-Log10(P-value)")
# グラフタイトル：腫瘍 vs 非腫瘍のタンパク質発現量差
ax.set_title("Differential Protein Abundance: Tumor vs Non-tumor")
# 凡例を右上に表示（枠なし）
ax.legend(frameon=False, loc="upper right")
# 上と右の枠線を非表示にしてすっきりした見た目にする
ax.spines[["top", "right"]].set_visible(False)

# 図をPNGファイルとして保存（dpi=150で高解像度、余白を最小化）
fig.savefig(f"{FIG_DIR}/fig_bonus_volcano.png", dpi=150, bbox_inches="tight")
# 図をJupyter上に表示
plt.show()

## 全有意差タンパク質 ヒートマップ + PCA

In [ ]:
def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """95% 信頼楕円を描画する。データ分布の広がりと相関を楕円で可視化。"""
    # データ点が2未満の場合は楕円を描画できないので終了
    if len(x) < 2:
        return
    # x, yの共分散行列を計算（分散・共分散の2x2行列）
    cov = np.cov(x, y)
    # ピアソン相関係数を計算（共分散を各標準偏差で割る）
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    # 楕円の横半径：相関が正なら大きくなる
    ell_rx = np.sqrt(1 + pearson)
    # 楕円の縦半径：相関が正なら小さくなる
    ell_ry = np.sqrt(1 - pearson)
    # 楕円パッチを作成（中心を原点に、幅と高さを設定）
    ellipse = Ellipse((0, 0), width=ell_rx * 2, height=ell_ry * 2, **kwargs)
    # アフィン変換を連鎖適用：45度回転 → 標準偏差でスケーリング → データの平均位置へ平行移動
    transf = (transforms.Affine2D()
              .rotate_deg(45)  # 45度回転して主軸方向に合わせる
              .scale(np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std)  # n_std倍の標準偏差でスケール
              .translate(np.mean(x), np.mean(y)))  # データの重心に移動
    # 楕円の座標変換を設定（データ座標系に変換）
    ellipse.set_transform(transf + ax.transData)
    # 楕円をAxesに追加して描画
    return ax.add_patch(ellipse)

In [ ]:
# --- 有意差タンパク質の抽出 ---
# 非有意（NS）を除外し、有意差のあるタンパク質のみを取得
sig_df = result_df[result_df["Significant"] != "NS"]
# 有意差タンパク質の発現量データを元のデータフレームから抽出
sig_data = df.loc[df.index.isin(sig_df["Protein"])]
# 各タンパク質のlog2FC値を取得（上昇/低下の方向判定に使用）
direction = sig_df.set_index("Protein")["Log2FC"]

# サンプルごとの色を条件に基づいて割り当て（ヒートマップの行カラーバー用）
sample_colors  = conditions.reindex(df.columns).map(COND_MAP)
# タンパク質ごとの色を発現方向に基づいて割り当て（ヒートマップの列カラーバー用）
# log2FC > 0 なら赤（Up）、それ以外は青（Down）
protein_colors = pd.Series(
    sig_data.index.map(lambda p: UP if direction.get(p, 0) > 0 else DOWN),
    index=sig_data.index, name="Expression"
)

# --- Figure 2a: 全有意差タンパク質のクラスタリングヒートマップ ---
g = sns.clustermap(
    sig_data.T,                    # データを転置（行:サンプル、列:タンパク質）
    method="ward",                 # Ward法で階層的クラスタリング（分散最小化）
    cmap="RdBu_r",                # 赤青カラーマップ（赤:高発現、青:低発現）
    z_score=1,                     # 列方向（タンパク質ごと）にzスコア正規化
    row_colors=sample_colors,      # 行（サンプル）の条件カラーバー
    col_colors=protein_colors,     # 列（タンパク質）の発現方向カラーバー
    xticklabels=False,             # x軸ラベル（タンパク質名）は非表示（多すぎるため）
    yticklabels=True,              # y軸ラベル（サンプル名）は表示
    figsize=(10, 8),               # 図のサイズ（幅10×高さ8インチ）
    vmin=-3, vmax=3,               # カラースケールの範囲（zスコア -3〜+3）
)
# y軸のサンプル名ラベルを小さめのフォントサイズに設定
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
# ヒートマップをPNG画像として保存
g.savefig(f"{FIG_DIR}/fig2a_heatmap_all.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 2b: 全有意差タンパク質を用いたPCA ---
pca = PCA(n_components=2)  # 2成分のPCAモデルを作成
# 有意差タンパク質の発現量で主成分分析を実行（転置してサンプル×タンパク質の形にする）
scores = pca.fit_transform(sig_data.T)

# PCAプロットの作成
fig, ax = plt.subplots(figsize=(8, 6))  # 8x6インチの図を作成
for cond, color in [("Normal", NORMAL), ("Tumor", TUMOR)]:
    # 各条件に属するサンプルのブールマスクを作成
    m = (conditions.reindex(df.columns) == cond).values
    # 散布図を描画：PC1 vs PC2、s=点サイズ、alpha=透明度、edgecolors=点の縁の色
    ax.scatter(scores[m, 0], scores[m, 1], c=color, s=80, alpha=0.8,
              label=cond, edgecolors="white")
    # 95%信頼楕円を描画（データ分布の広がりを楕円で可視化）
    confidence_ellipse(scores[m, 0], scores[m, 1], ax,
                       facecolor=color, alpha=0.15, edgecolor=color, lw=1.5)
# x軸ラベル：第1主成分と寄与率（%）
ax.set_xlabel(f"Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
# y軸ラベル：第2主成分と寄与率（%）
ax.set_ylabel(f"Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
# グラフタイトル：全有意差タンパク質を使ったPCA
ax.set_title("PCA using All Differentially Abundant Proteins")
# 凡例を表示（枠なし）
ax.legend(frameon=False)
# 上と右の枠線を非表示
ax.spines[["top", "right"]].set_visible(False)
# グリッド線を半透明の破線で表示
ax.grid(True, alpha=0.3, ls="--")
# PCAプロットをPNG画像として保存
fig.savefig(f"{FIG_DIR}/fig2b_pca_all.png", dpi=150, bbox_inches="tight")
plt.show()

## Top N 解析

In [ ]:
def plot_top_n(df, result_df, conditions):
    """Top 50/100/200 の clustering + PCA を一括描画する。"""
    # 有意差のあるタンパク質のみを抽出（NSを除外）
    sig = result_df[result_df["Significant"] != "NS"].copy()
    # log2FCの絶対値列を追加（発現変化量の大きさでソートするため）
    sig["AbsLog2FC"] = sig["Log2FC"].abs()
    # 上昇タンパク質を発現変化量の大きい順にソート
    sig_up   = sig.query("Significant == 'Up'").sort_values("AbsLog2FC", ascending=False)
    # 低下タンパク質を発現変化量の大きい順にソート
    sig_down = sig.query("Significant == 'Down'").sort_values("AbsLog2FC", ascending=False)

    # 各タンパク質のlog2FC値（発現方向の判定に使用）
    direction = sig.set_index("Protein")["Log2FC"]
    # サンプルごとの条件カラー（ヒートマップの行カラーバー用）
    sample_colors = conditions.reindex(df.columns).map(COND_MAP)
    # ヒートマップの図番号ラベル（Fig 2c, 2d, 2e）
    hm_label = {50: "c", 100: "d", 200: "e"}
    # PCAの図番号ラベル（Fig 2f, 2g, 2h）
    pc_label = {50: "f", 100: "g", 200: "h"}

    # Top 50, 100, 200 それぞれについてヒートマップとPCAを作成
    for n in [50, 100, 200]:
        # 上昇Top N個と低下Top N個のタンパク質名を結合（実際の個数が足りない場合はある分だけ使用）
        top_proteins = np.concatenate([
            sig_up.head(min(n, len(sig_up)))["Protein"].values,
            sig_down.head(min(n, len(sig_down)))["Protein"].values,
        ])
        # 選択したタンパク質の発現量データを抽出
        top_data = df.loc[df.index.isin(top_proteins)]
        # タンパク質ごとの発現方向カラー（ヒートマップの列カラーバー用）
        prot_colors = pd.Series(
            top_data.index.map(lambda p: UP if direction.get(p, 0) > 0 else DOWN),
            index=top_data.index, name="Expression"
        )

        # クラスタリングヒートマップの描画（Fig 2c-e）
        g = sns.clustermap(
            top_data.T,                    # 転置（行:サンプル、列:タンパク質）
            method="ward",                 # Ward法で階層的クラスタリング
            cmap="RdBu_r",                # 赤青カラーマップ
            z_score=1,                     # 列方向にzスコア正規化
            row_colors=sample_colors,      # 行カラーバー（Normal/Tumor）
            col_colors=prot_colors,        # 列カラーバー（Up/Down）
            xticklabels=False,             # タンパク質名は非表示
            yticklabels=True,              # サンプル名は表示
            figsize=(8, 10),               # 図のサイズ（幅8×高さ10インチ）
            vmin=-3, vmax=3,               # カラースケール範囲
        )
        # y軸ラベルのフォントサイズを小さく設定
        g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
        # ヒートマップを保存（ファイル名にTop Nの数を含む）
        g.savefig(f"{FIG_DIR}/fig2{hm_label[n]}_clustering_top{n}.png",
                  dpi=150, bbox_inches="tight")
        plt.show()

        # PCAプロットの描画（Fig 2f-h）
        pca = PCA(n_components=2)  # 2成分PCAモデルを作成
        # 選択したTop Nタンパク質の発現量でPCAを実行
        scores = pca.fit_transform(top_data.T)
        fig, ax = plt.subplots(figsize=(8, 6))  # 8x6インチの図を作成
        for cond, color in [("Normal", NORMAL), ("Tumor", TUMOR)]:
            # 条件ごとのサンプルマスクを作成
            m = (conditions.reindex(df.columns) == cond).values
            # 散布図を描画：s=点サイズ80、白い縁取り付き
            ax.scatter(scores[m, 0], scores[m, 1], c=color, s=80, alpha=0.8,
                      label=cond, edgecolors="white")
            # 95%信頼楕円を描画（半透明の塗りつぶし+境界線）
            confidence_ellipse(scores[m, 0], scores[m, 1], ax,
                               facecolor=color, alpha=0.15, edgecolor=color, lw=1.5)
        # x軸ラベル：第1主成分と寄与率
        ax.set_xlabel(f"Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        # y軸ラベル：第2主成分と寄与率
        ax.set_ylabel(f"Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        # タイトル：使用したTop Nの数を表示
        ax.set_title(f"PCA using Top {n} Differentially Abundant Proteins")
        # 凡例を表示（枠なし）
        ax.legend(frameon=False)
        # 上と右の枠線を非表示
        ax.spines[["top", "right"]].set_visible(False)
        # グリッド線を半透明の破線で表示
        ax.grid(True, alpha=0.3, ls="--")
        # PCAプロットを保存
        fig.savefig(f"{FIG_DIR}/fig2{pc_label[n]}_pca_top{n}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()
        # 実際に使用したタンパク質数を表示
        print(f"Top {n}: {len(top_data)} proteins")

In [ ]:
# Top 50/100/200の有意差タンパク質について、ヒートマップとPCAプロットを一括生成
plot_top_n(df, result_df, conditions)